<a href="https://colab.research.google.com/github/Zattyla/K-UVR-Colab/blob/main/K-UVR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌊 UVR Elite Audio Separator (v1.0)
*Advanced Vocal & Instrumental Extraction Pipeline*

### 🚀 How to use:
1.  **Select GPU:** Ensure you are using a **T4 GPU** (`Runtime` -> `Change runtime type`).
2.  **Run Cell 1:** Install the core engine and dependencies.
3.  **Run Cell 2:** Download the professional model library (Kim, MDX-Net, etc.).
4.  **Upload Audio:** Click the **Folder icon** on the left and upload your song to the `inputs` folder.
5.  **Run Cell 3:** Enter your filename, choose a model, and toggle **DEREVERB** for cleaner AI covers.

### 🔬 Model Guide:
- **Kim_Vocal_2:** Your "Go-to" model. Incredible vocal isolation.
- **MDX_UVR_Vocal_FT:** Use this if the song has heavy backing vocals or complex harmonies.
- **UVR_Inst_HQ_3:** Use this to get a studio-quality instrumental track for karaoke or mixing.
- **De-reverb:** Automatically cleans the "hall effect" or echo from the vocals.

---

In [ ]:
# @title 1. Setup & Dependencies
import os
print("⏳ Installing UVR Engine...")

# Install latest audio-separator with GPU support
!pip install -U numpy>=2.0.0
!pip install -q audio-separator[gpu]

# Create workspace
os.makedirs("/content/inputs", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)

print("✅ UVR Environment Ready! Upload your songs to the 'inputs' folder.")

In [ ]:
# @title 2. Download Multi-Model Library
import os

# Model dictionary: Name -> Filename
models = {
    "Kim_Vocal_2": "Kim_Vocal_2.onnx", # Best for overall Vocals
    "MDX_UVR_Vocal_FT": "UVR-MDX-NET-Voc_FT.onnx", # Best for clean Backing Vocals
    "UVR_Inst_HQ_3": "UVR-MDX-NET-Inst_HQ_3.onnx", # Best for High Quality Instrumentals
    "Reverb_HQ": "UVR-DeReverb-HQ.onnx" # Best for removing Reverb/Echo
}

print("⏬ Downloading model library...")

for name, file in models.items():
    model_path = f"/content/models/{file}"
    if not os.path.exists(model_path):
        # Base URL for UVR models on HuggingFace
        url = f"https://huggingface.co/lucasnewman/uvr-models/resolve/main/MDX-Net/{file}"
        print(f"Downloading {name}...")
        os.system(f'curl -L -o "{model_path}" "{url}"')

print("✅ All models are ready!")

In [ ]:
# @title 3. Start Separation
# @markdown ### Input Settings
FILE_NAME = "song.mp3" # @param {type:"string"}

# @markdown ### Select Model Strategy
# @markdown - **Kim_Vocal_2**: Standard high-quality vocal extraction.
# @markdown - **MDX_UVR_Vocal_FT**: Extra clean vocals (Fine-tuned).
# @markdown - **UVR_Inst_HQ_3**: Best if you only care about the instrumental.
MODEL_CHOICE = "Kim_Vocal_2" # @param ["Kim_Vocal_2", "MDX_UVR_Vocal_FT", "UVR_Inst_HQ_3"]

# @markdown ### Post-Processing
DEREVERB = True # @param {type:"boolean"}

input_path = f"/content/inputs/{FILE_NAME}"
model_file = models[MODEL_CHOICE]

if os.path.exists(input_path):
    print(f"🚀 Processing {FILE_NAME} with {MODEL_CHOICE}...")

    # Main Separation
    !audio-separator "{input_path}" \
        --model_filename {model_file} \
        --output_dir "/content/outputs" \
        --output_format WAV \
        --normalization 0.9

    # Optional De-reverb (makes RVC much better)
    if DEREVERB:
        print("🧹 Running De-reverb for cleaner vocals...")
        # Find the vocal file just created (it usually has 'Vocals' in the name)
        !audio-separator "/content/outputs/"*"Vocals"*".wav" \
            --model_filename UVR-DeReverb-HQ.onnx \
            --output_dir "/content/outputs" \
            --output_format WAV

    print("✨ Finished! Check the 'outputs' folder.")
else:
    print(f"❌ Error: {FILE_NAME} not found in /content/inputs")